# DINOv3 Fine-tuning on Custom Data

> Fine-tune Meta's **DINOv3** (Distillation with No Labels v3) Vision Transformer on your own image classification dataset.

DINOv3 is Meta's 2025 state-of-the-art self-supervised ViT, trained on ~1.7 billion images with a 7B-parameter teacher model.
This notebook demonstrates three fine-tuning strategies:

1. **Linear Probe** — freeze backbone, train only a linear head (fastest, least compute)
2. **Full Fine-tune** — unfreeze and update all weights (highest accuracy, most compute)
3. **LoRA Fine-tune** — low-rank adaptation of encoder blocks (best accuracy/compute trade-off)

### Requirements
- `transformers >= 4.56.0` (first version with DINOv3 support)
- `timm >= 1.0.20`
- `torch >= 2.0`
- A Hugging Face account with access to gated DINOv3 weights

## 1. Installation

In [ ]:
# Install / upgrade required libraries
# DINOv3 support requires transformers >= 4.56.0
!pip install -q --upgrade \
    'transformers>=4.56.0' \
    'timm>=1.0.20' \
    'torch>=2.0' \
    torchvision \
    huggingface_hub \
    peft \
    matplotlib \
    scikit-learn \
    tqdm

## 2. Imports & Global Config

In [ ]:
import os
import random
import math
import copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as T
from torchvision.datasets import ImageFolder

from transformers import (
    AutoImageProcessor,
    AutoModel,
    Dinov3Model,
    Dinov3Config,
)
from huggingface_hub import login

print(f"PyTorch      : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device : {device}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── Global configuration — edit these to match your setup ─────────────────────
CFG = dict(
    # ---- Data ----------------------------------------------------------------
    # Root folder that contains one sub-folder per class:
    #   data_root/
    #       class_a/  img1.jpg  img2.png ...
    #       class_b/  img1.jpg  ...
    data_root       = "./data",          # <-- change to your dataset path
    val_split       = 0.15,              # fraction used for validation
    test_split      = 0.10,             # fraction used for test
    num_workers     = 4,

    # ---- Model ---------------------------------------------------------------
    # Available DINOv3 checkpoints on Hugging Face:
    #   facebook/dinov3-vits16-pretrain-lvd1689m   (ViT-S, 21M params)
    #   facebook/dinov3-vitb16-pretrain-lvd1689m   (ViT-B, 86M params)  ← default
    #   facebook/dinov3-vitl16-pretrain-lvd1689m   (ViT-L, 300M params)
    #   facebook/dinov3-vith16-pretrain-lvd1689m   (ViT-H+, 840M params)
    model_name      = "facebook/dinov3-vitb16-pretrain-lvd1689m",
    num_classes     = None,             # auto-detected from folder structure

    # ---- Fine-tuning strategy: 'linear_probe' | 'full' | 'lora' -------------
    strategy        = "linear_probe",

    # ---- LoRA settings (used only when strategy == 'lora') ------------------
    lora_rank       = 8,
    lora_alpha      = 16,
    lora_dropout    = 0.05,

    # ---- Training ------------------------------------------------------------
    image_size      = 224,
    batch_size      = 32,
    epochs          = 20,
    lr              = 1e-3,             # head learning rate
    backbone_lr     = 1e-5,            # backbone lr (used for 'full' strategy)
    weight_decay    = 1e-4,
    patience        = 5,               # early-stopping patience
    output_dir      = "./dinov3_finetuned",
)

os.makedirs(CFG['output_dir'], exist_ok=True)
print("Config loaded:", CFG)

## 3. Authenticate with Hugging Face

DINOv3 weights are **gated**. Accept the license at  
https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m  
then paste your token below (or set the `HF_TOKEN` environment variable).

In [ ]:
import os

hf_token = os.environ.get("HF_TOKEN", None)
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in via HF_TOKEN env var.")
else:
    # Interactive login — will prompt for token
    login()

## 4. Dataset Preparation

Expects an **ImageFolder**-style directory layout:
```
data/
├── cat/
│   ├── cat_001.jpg
│   └── ...
└── dog/
    ├── dog_001.jpg
    └── ...
```

In [ ]:
# ── Image transforms ──────────────────────────────────────────────────────────
IMG_MEAN = [0.485, 0.456, 0.406]   # ImageNet statistics (DINOv3 was trained with these)
IMG_STD  = [0.229, 0.224, 0.225]

train_transforms = T.Compose([
    T.RandomResizedCrop(CFG['image_size'], scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

val_transforms = T.Compose([
    T.Resize(int(CFG['image_size'] * 1.14)),  # resize short side
    T.CenterCrop(CFG['image_size']),
    T.ToTensor(),
    T.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

In [ ]:
# ── Build dataset splits ──────────────────────────────────────────────────────
full_dataset = ImageFolder(root=CFG['data_root'], transform=train_transforms)

CFG['num_classes'] = len(full_dataset.classes)
class_names = full_dataset.classes
print(f"Found {len(full_dataset)} images across {CFG['num_classes']} classes: {class_names}")

n_total = len(full_dataset)
n_val   = int(n_total * CFG['val_split'])
n_test  = int(n_total * CFG['test_split'])
n_train = n_total - n_val - n_test

train_ds, val_ds, test_ds = random_split(
    full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

# Apply val transforms to val/test splits
val_ds.dataset  = copy.copy(full_dataset)
val_ds.dataset.transform  = val_transforms
test_ds.dataset = copy.copy(full_dataset)
test_ds.dataset.transform = val_transforms

print(f"Train: {n_train} | Val: {n_val} | Test: {n_test}")

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)

In [ ]:
# ── Visualise a few training samples ─────────────────────────────────────────
def denormalize(tensor, mean=IMG_MEAN, std=IMG_STD):
    t = tensor.clone()
    for c, m, s in zip(range(3), mean, std):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    if i >= len(imgs): break
    ax.imshow(denormalize(imgs[i]).permute(1, 2, 0).numpy())
    ax.set_title(class_names[labels[i].item()], fontsize=9)
    ax.axis('off')
plt.suptitle("Sample training images", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Load DINOv3 Backbone

We load the **DINOv3** backbone from Hugging Face using `Dinov3Model`.  
The model outputs a `[CLS]` token embedding that we use as the image representation.

In [ ]:
# Load the image processor (handles resizing / normalisation consistently with training)
processor = AutoImageProcessor.from_pretrained(CFG['model_name'])

# Load the DINOv3 backbone
backbone = Dinov3Model.from_pretrained(CFG['model_name'])

# Determine the embedding dimension from the config
embed_dim = backbone.config.hidden_size
print(f"Model    : {CFG['model_name']}")
print(f"Embed dim: {embed_dim}")
print(f"Params   : {sum(p.numel() for p in backbone.parameters()) / 1e6:.1f}M")

## 6. Build the Classification Model

We wrap the DINOv3 backbone with a lightweight classification head.

In [ ]:
class Dinov3Classifier(nn.Module):
    """DINOv3 backbone + linear classification head.

    The head receives the concatenation of:
      - the [CLS] token
      - the mean-pooled patch tokens
    giving an effective feature dim of 2 * embed_dim.
    """

    def __init__(self, backbone: Dinov3Model, num_classes: int, dropout: float = 0.3):
        super().__init__()
        self.backbone = backbone
        embed_dim = backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim * 2),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, num_classes),
        )

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        outputs = self.backbone(pixel_values=pixel_values)
        # last_hidden_state: (batch, seq_len, embed_dim)
        # index 0 → [CLS], 1: → patch tokens
        cls_token    = outputs.last_hidden_state[:, 0]       # (B, D)
        patch_tokens = outputs.last_hidden_state[:, 1:].mean(dim=1)  # (B, D)
        features     = torch.cat([cls_token, patch_tokens], dim=1)   # (B, 2D)
        return self.head(features)

In [ ]:
def build_model(strategy: str) -> nn.Module:
    """Return a Dinov3Classifier configured for the requested strategy."""

    if strategy == 'linear_probe':
        # Freeze the entire backbone — only the head is trainable
        for p in backbone.parameters():
            p.requires_grad = False
        model = Dinov3Classifier(backbone, CFG['num_classes'])
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in model.parameters())
        print(f"[linear_probe] Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M params")

    elif strategy == 'full':
        # All parameters trainable; backbone uses a lower LR (set in optimizer)
        for p in backbone.parameters():
            p.requires_grad = True
        model = Dinov3Classifier(backbone, CFG['num_classes'])
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in model.parameters())
        print(f"[full] Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M params")

    elif strategy == 'lora':
        from peft import get_peft_model, LoraConfig, TaskType

        # Freeze backbone first
        for p in backbone.parameters():
            p.requires_grad = False

        # Identify attention projection layers to apply LoRA to
        target_modules = ["query", "key", "value", "dense"]

        lora_cfg = LoraConfig(
            task_type      = TaskType.FEATURE_EXTRACTION,
            r              = CFG['lora_rank'],
            lora_alpha     = CFG['lora_alpha'],
            lora_dropout   = CFG['lora_dropout'],
            target_modules = target_modules,
            bias           = "none",
        )
        lora_backbone = get_peft_model(backbone, lora_cfg)
        lora_backbone.print_trainable_parameters()

        # Build classifier with LoRA-wrapped backbone
        model = Dinov3Classifier(lora_backbone, CFG['num_classes'])
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in model.parameters())
        print(f"[lora] Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M params")

    else:
        raise ValueError(f"Unknown strategy '{strategy}'. Choose: linear_probe | full | lora")

    return model.to(device)


model = build_model(CFG['strategy'])
print(model)

## 7. Optimizer, Scheduler & Loss

In [ ]:
def build_optimizer(model: nn.Module, strategy: str) -> optim.Optimizer:
    if strategy == 'full':
        # Separate learning rates for backbone and head
        backbone_params = [p for n, p in model.named_parameters()
                           if 'backbone' in n and p.requires_grad]
        head_params     = [p for n, p in model.named_parameters()
                           if 'head' in n and p.requires_grad]
        param_groups = [
            {'params': backbone_params, 'lr': CFG['backbone_lr']},
            {'params': head_params,     'lr': CFG['lr']},
        ]
    else:
        param_groups = [p for p in model.parameters() if p.requires_grad]

    return optim.AdamW(param_groups, lr=CFG['lr'], weight_decay=CFG['weight_decay'])


criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = build_optimizer(model, CFG['strategy'])

# Cosine annealing with linear warm-up
warmup_epochs = max(1, CFG['epochs'] // 10)
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs),
        optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'] - warmup_epochs, eta_min=1e-7),
    ],
    milestones=[warmup_epochs],
)

print(f"Optimizer  : AdamW | LR {CFG['lr']} | WD {CFG['weight_decay']}")
print(f"Scheduler  : LinearWarmup ({warmup_epochs} ep) → CosineAnnealing")

## 8. Training Loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, phase='train'):
    """One epoch of training or validation. Returns (loss, accuracy)."""
    is_train = (phase == 'train')
    model.train(is_train)
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for imgs, labels in tqdm(loader, desc=phase, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)

            logits = model(imgs)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)

    return total_loss / total, correct / total

In [ ]:
history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
no_improve   = 0
best_ckpt    = os.path.join(CFG['output_dir'], 'best_model.pt')

print(f"\nTraining DINOv3 [{CFG['strategy']}] for {CFG['epochs']} epochs ...\n")

for epoch in range(1, CFG['epochs'] + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, 'train')
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion, None,      'val')
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    lr_now = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:>3}/{CFG['epochs']} "
          f"| LR {lr_now:.2e} "
          f"| Train loss {train_loss:.4f}  acc {train_acc:.4f} "
          f"| Val   loss {val_loss:.4f}  acc {val_acc:.4f}", end="")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        no_improve   = 0
        torch.save(model.state_dict(), best_ckpt)
        print("  ← best")
    else:
        no_improve += 1
        print(f"  (no improvement {no_improve}/{CFG['patience']})")

    if no_improve >= CFG['patience']:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nBest val accuracy: {best_val_acc:.4f}")
print(f"Checkpoint saved : {best_ckpt}")

## 9. Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_ran, history['train_loss'], label='Train')
axes[0].plot(epochs_ran, history['val_loss'],   label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_ran, history['train_acc'], label='Train')
axes[1].plot(epochs_ran, history['val_acc'],   label='Val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle(f'DINOv3 Fine-tune ({CFG["strategy"]}) — {CFG["model_name"].split("/")[-1]}')
plt.tight_layout()
plt.savefig(os.path.join(CFG['output_dir'], 'training_curves.png'), dpi=150)
plt.show()

## 10. Evaluation on Test Set

In [ ]:
# Reload best checkpoint
model.load_state_dict(torch.load(best_ckpt, map_location=device))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Evaluating test set'):
        imgs   = imgs.to(device)
        logits = model(imgs)
        preds  = logits.argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

test_acc = (all_preds == all_labels).mean()
print(f"\nTest Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# Confusion matrix
import itertools

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(max(6, CFG['num_classes']), max(5, CFG['num_classes'] - 1)))
im = ax.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(CFG['num_classes']))
ax.set_yticks(range(CFG['num_classes']))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
thresh = cm_norm.max() / 2
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]:.2f})",
            ha='center', va='center',
            color='white' if cm_norm[i,j] > thresh else 'black', fontsize=8)
ax.set_ylabel('True label')
ax.set_xlabel('Predicted label')
ax.set_title('Confusion Matrix (normalised)')
plt.tight_layout()
plt.savefig(os.path.join(CFG['output_dir'], 'confusion_matrix.png'), dpi=150)
plt.show()

## 11. Inference on a Single Image

In [ ]:
def predict_image(image_path: str, model: nn.Module, transform, top_k: int = 3):
    """Run inference on a single image and return top-k predictions."""
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0]

    topk_probs, topk_idxs = probs.topk(min(top_k, len(class_names)))
    results = [(class_names[i.item()], p.item()) for i, p in zip(topk_idxs, topk_probs)]

    # Display
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img)
    axes[0].set_title('Input image')
    axes[0].axis('off')

    labels_  = [r[0] for r in results]
    scores_  = [r[1] for r in results]
    axes[1].barh(labels_[::-1], scores_[::-1])
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel('Probability')
    axes[1].set_title(f'Top-{top_k} Predictions')

    plt.tight_layout()
    plt.show()
    return results


# Example — replace with your image path
# results = predict_image('./my_image.jpg', model, val_transforms)
# print(results)

## 12. Save & Export the Fine-tuned Model

In [ ]:
# ── Save full model weights ───────────────────────────────────────────────────
final_ckpt = os.path.join(CFG['output_dir'], 'dinov3_classifier_final.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names'     : class_names,
    'num_classes'     : CFG['num_classes'],
    'strategy'        : CFG['strategy'],
    'model_name'      : CFG['model_name'],
    'embed_dim'       : embed_dim,
    'best_val_acc'    : best_val_acc,
}, final_ckpt)
print(f"Final checkpoint saved: {final_ckpt}")

# ── (Optional) Push backbone to Hugging Face Hub ─────────────────────────────
# If you want to share the fine-tuned backbone:
#
# backbone.push_to_hub("your-hf-username/dinov3-finetuned-custom")
# processor.push_to_hub("your-hf-username/dinov3-finetuned-custom")

In [ ]:
# ── Load checkpoint for downstream use ───────────────────────────────────────
def load_dinov3_classifier(checkpoint_path: str) -> nn.Module:
    """Re-instantiate and load a saved DINOv3 classifier."""
    ckpt = torch.load(checkpoint_path, map_location='cpu')

    _backbone = Dinov3Model.from_pretrained(ckpt['model_name'])
    _model    = Dinov3Classifier(_backbone, ckpt['num_classes'])
    _model.load_state_dict(ckpt['model_state_dict'])
    _model.eval()

    print(f"Loaded model | classes: {ckpt['class_names']} | val acc: {ckpt['best_val_acc']:.4f}")
    return _model, ckpt['class_names']


# loaded_model, loaded_classes = load_dinov3_classifier(final_ckpt)
print("Load helper defined. Uncomment above lines to test reloading.")

## 13. Using timm as an Alternative Backend

If you prefer **timm** (requires `timm >= 1.0.20`), the DINOv3 backbone can be loaded as follows.  
This is a drop-in replacement for the HF `Dinov3Model` loading above.

In [ ]:
# timm alternative (commented out — uncomment to use instead of the HF approach)
"""
import timm

# Available DINOv3 timm model names:
#   vit_small_patch16_224.dinov3   (21M)
#   vit_base_patch16_224.dinov3    (86M)  ← default
#   vit_large_patch16_224.dinov3   (300M)
#   vit_huge_patch16_224.dinov3    (840M)

timm_model_name = 'vit_base_patch16_224.dinov3'

# Load as a feature extractor (num_classes=0 removes the classification head)
timm_backbone = timm.create_model(timm_model_name, pretrained=True, num_classes=0)
embed_dim = timm_backbone.embed_dim

class Dinov3ClassifierTimm(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.LayerNorm(backbone.embed_dim),
            nn.Dropout(dropout),
            nn.Linear(backbone.embed_dim, num_classes),
        )
    def forward(self, x):
        features = self.backbone(x)   # global avg pool output
        return self.head(features)

timm_model = Dinov3ClassifierTimm(timm_backbone, CFG['num_classes']).to(device)
print(timm_model)
"""

## Summary

| Strategy | Trainable params | Compute | Typical accuracy |
|---|---|---|---|
| `linear_probe` | ~1–4 K (head only) | Very low | Good baseline |
| `lora` | ~1–3 M (LoRA + head) | Low–medium | Near full fine-tune |
| `full` | All (~86–840 M) | High | Highest, risk of overfitting |

**Tips:**
- Start with `linear_probe` to establish a baseline quickly.
- Move to `lora` for best accuracy/compute trade-off with smaller datasets (< 10 k images).
- Use `full` fine-tuning only with large datasets and sufficient GPU memory.
- DINOv3 features benefit from **concatenating CLS + mean-patch tokens** (done above).
- DINOv3 weights are **gated on Hugging Face** — remember to accept the license first.

**References:**
- [DINOv3 GitHub (facebookresearch/dinov3)](https://github.com/facebookresearch/dinov3)
- [Hugging Face DINOv3 docs](https://huggingface.co/docs/transformers/main/model_doc/dinov3)
- [dinov3-finetune LoRA example](https://github.com/RobvanGastel/dinov3-finetune)
- [DINOv3 Custom Training Framework](https://github.com/wpawgasa/DINOv3_custom_training)